<a href="https://colab.research.google.com/github/profcomff/chatbot-mark-api/blob/dev_fedor/notebooks/Create_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !rm -rf /content/chatbot-mark-api

In [1]:
# !git clone https://github.com/profcomff/chatbot-mark-api.git
!git clone --branch dev_fedor https://github.com/profcomff/chatbot-mark-api.git

Cloning into 'chatbot-mark-api'...
remote: Enumerating objects: 441, done.
remote: Counting objects: 100% (272/272), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 441 (delta 131), reused 196 (delta 81), pack-reused 169 (from 1)
Receiving objects: 100% (441/441), 4.55 MiB | 11.78 MiB/s, done.
Resolving deltas: 100% (183/183), done.


# Библиотеки


In [2]:
!pip install langchain transformers sentence-transformers -q
!pip install -U langchain-community -q
!pip install -qU "langchain-chroma>=0.1.2" -q
!pip install langchain_huggingface -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00


In [3]:
from tqdm import tqdm

import numpy as np
import pandas as pd

from transformers import XLMRobertaTokenizer, XLMRobertaModel
import torch

from langchain.schema import Document

from langchain_chroma import Chroma

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

#для препроцессинга key_words
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Функции/классы

In [8]:
import sys
sys.path.append("/content/chatbot-mark-api")

from search.nn import E5LangChainEmbedder

In [9]:
def safe_add_documents(vector_store, chunks, chroma_batch_size=1000):
    with tqdm(total=len(chunks), desc="Добавление в Chroma", unit="doc") as pbar:
        for i in range(0, len(chunks), chroma_batch_size):
            try:
                batch = chunks[i:i+chroma_batch_size]
                vector_store.add_documents(batch)
                pbar.update(len(batch))
            except Exception as e:
                if "Batch size" in str(e) and "greater than max" in str(e):
                    new_size = chroma_batch_size // 2
                    print(f"Ошибка: {e}. Уменьшаю размер батча до {new_size}")
                    return safe_add_documents(vector_store, chunks[i:], new_size)
                raise
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    print("Все документы успешно добавлены!")

# 0. Загрузка контекстов / скачивание модели


In [10]:
# answers = pd.read_excel('/content/chatbot-mark-api/file/database_v2.xlsx') #!!! should change
# answers = pd.read_excel('/content/chatbot-mark-api/file/database_v2_key_words.xlsx')
answers = pd.read_excel('/content/chatbot-mark-api/file/database_v4_key_words.xlsx')

display(answers.answer[0])
display(answers.head(2))

'Карта зачет. https://vk.com/wall-24234717_22977\nЭто ваш профсоюзный билет. С помощью этой карты вы можете получать скидки у полезных для студентов популярных брендов, участвовать в конкурсах и розыгрышах, а также посещать концерты и мероприятия. \nПолный перечень скидок есть в статье: vk.cc/bYSCNw.'

,Unnamed: 0,topic_name,answer,id,Key words,structure
0,0,Карта зачет,Карта зачет. https://vk.com/wall-24234717_2297...,0,Карта зачет,NaN
1,1,Как вступить в профсоюз,Как вступить в профсоюз? Чтобы вступить в Проф...,1,Профсоюз,NaN


link to model in HuggingFace [e5-base-en-ru](https://huggingface.co/d0rj/e5-base-en-ru)

In [11]:
tokenizer = XLMRobertaTokenizer.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)
search_model = XLMRobertaModel.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/471 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/529M [00:00<?, ?B/s]

# 1. Создание БД c помощью e5


In [19]:
all_chunks = []

for answer, topic_name, kw in zip(answers['answer'], answers['topic_name'], answers['Key words']):
    all_chunks.append(Document(
        page_content=answer,
        metadata={
            "source": topic_name.strip(),
            "key_words": kw,
        }
    ))

# Инициализация эмбеддера E5
embedder = E5LangChainEmbedder(
    tokenizer=tokenizer,
    model=search_model,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    add_prefix=True,  #!!!
    disable_tqdm=False,
)

# Создание или загрузка векторного хранилища Chroma
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embedder,
    persist_directory="./chroma_db"  #!!!
)

# Безопасное добавление документов в векторное хранилище
safe_add_documents(vector_store, all_chunks)

Добавление в Chroma: 100%|██████████| 98/98 [01:22<00:00,  1.19doc/s]

Все документы успешно добавлены!


In [18]:
!zip -r chroma_db.zip chroma_db/

  adding: chroma_db/ (stored 0%)
  adding: chroma_db/d5d200f8-dd9e-4ec9-9e6f-c56182165e7c/ (stored 0%)
  adding: chroma_db/d5d200f8-dd9e-4ec9-9e6f-c56182165e7c/header.bin (deflated 61%)
  adding: chroma_db/d5d200f8-dd9e-4ec9-9e6f-c56182165e7c/data_level0.bin (deflated 100%)
  adding: chroma_db/d5d200f8-dd9e-4ec9-9e6f-c56182165e7c/length.bin (deflated 100%)
  adding: chroma_db/d5d200f8-dd9e-4ec9-9e6f-c56182165e7c/link_lists.bin (stored 0%)
  adding: chroma_db/chroma.sqlite3 (deflated 54%)


# Подключение БД

In [13]:
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embedder,
    persist_directory="./chroma_db"
)

In [14]:
query = "профком?"

relevant_docs = vector_store.similarity_search(
    query,
    k=3,
)

In [15]:
relevant_docs

[Document(id='c7fb95f7-a82d-48dd-bdb4-753bc4da79ea', metadata={'key_words': 'Профком', 'source': 'Режим работы профкома'}, page_content='Режим работы профкома. Режим работы Профкома.  Часы работы: 11:00-16:00, ПН-ПТ. \nКабинет 2-39'),
 Document(id='ef777c02-d4ae-4c1e-a542-9caaa17ab617', metadata={'key_words': 'Профком, группа профкома', 'source': 'Связь с профкомом'}, page_content='Связь с профкомом. Как связаться с Профкомом? Вы можете написать нам в личных сообщениях в группе Профкома ВКонтакте (https://vk.com/profcomff), а также в нашем телеграмм-канале (https://t.me/profcom_ff).'),
 Document(id='4762f096-4cc3-4c63-91f1-1b5713ac3fb1', metadata={'source': 'Начало работы в профкоме', 'key_words': 'Профком'}, page_content='Начало работы в профкоме. Как начать что-то делать в Профкоме? Чтобы стать активистом Профкома, можно написать в личные сообщения группы https://vk.com/profcomff, прийти лично в кабинет Профкома (2-39) или прийти на любое наше мероприятие и познакомиться с представит

In [16]:
if False:
    # Получаем все документы с метаданными
    all_docs = vector_store.get(include=["metadatas"], limit=100)

    # Просматриваем key_words из метаданных
    for i, metadata in enumerate(all_docs["metadatas"]):
        if "key_words" in metadata:
            print(f"Документ {i}: {metadata['key_words']}")

    print(i)

In [17]:
# Собрать все уникальные key_words
all_keywords = set()
all_docs = vector_store.get(include=["metadatas"])

for metadata in all_docs["metadatas"]:
    if "key_words" in metadata and metadata["key_words"]:
        # Разделяем составные ключевые слова
        keywords = [kw.strip() for kw in str(metadata["key_words"]).split(",") if kw.strip()]
        all_keywords.update(keywords)

print("Все уникальные key_words:")
for kw in sorted(all_keywords):
    print(f"- {kw}")

Все уникальные key_words:
- Академ
- БДНС
- График
- КСД
- Каникулы
- Карта  зачет
- Лагеря
- МФК
- Машина
- Общага
- Перевод
- Пересдачи
- Праздники
- Принтер
- Профком
- Профсоюз
- РЖД
- Социальная карта
- Справка
- Стипендии
- Театр
- Экзамен
- Этажи
- академический отпуск
- выходные
- группа профкома
- кабинет
- календарь
- каникулы
- мфк
- общежитие
- перевод на бюджет
- пропуск
- профком
- рабочие дни
